# CGM Sandbox — JupyterHealth Exchange Data

This notebook runs the **CGM Sandbox** against data in the shape the
JupyterHealth Exchange client returns, using the synthetic sample in
`sample_data/jhe/`. Nothing here needs credentials or a network connection, so
it is the quickest way to confirm the sandbox works before pointing it at a
real study.

The sample mirrors study `30006` ("CGM & Wearables Demo"): one participant, five
days of CGM, plus sleep, food, heart rate and oxygen saturation. Every modality
the sandbox can draw is present, except sleep *stage episodes* — see the note at
the end for why that matters.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import pandas as pd

from cgmsandbox import (
    CGMViewer,
    FoodEntryOverlay,
    SleepCompositionExtension,
    SleepWindowOverlay,
    TimeInRangeOverlay,
    load_cgm_data,
    load_food_entry_data,
    load_sleep_nights,
    nocturnal_vs_diurnal,
)

DATA = "./sample_data/jhe"


def read_sample(modality):
    """Read one modality of the synthetic JHE sample.

    These CSVs are the client frame flattened to one row per record, so
    timestamps arrive as strings. The client returns them tz-aware, so they are
    parsed back to UTC here. Only the non-`_local` columns are parsed: the
    `_local` ones are already in the participant's clock time.
    """
    df = pd.read_csv(f"{DATA}/{modality}.csv")
    for col in df.columns:
        if col.endswith("date_time"):
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df

## Loading CGM Data

`load_cgm_data(source="client", ...)` expects the frame you would get back from
`jh_client.list_observations_df(..., code=Code.BLOOD_GLUCOSE)`. Because the
sample is already flattened to that shape, reading the CSV is the only setup
step.

In [ ]:
cgm_raw = read_sample("blood_glucose")

cgm_df = load_cgm_data(source="client", client_df=cgm_raw)
cgm_df.head()

## Loading Food Entry Data

Food entries are drawn as markers on the glucose trace, which makes it easy to
see a post-meal rise in context.

In [ ]:
food_raw = read_sample("food_entry")

food_df = load_food_entry_data(source="client", client_df=food_raw)
food_df.head()

## Loading Sleep Data

Sleep in this study is reported as **one aggregate record per night** — total,
deep, light and REM durations plus sleep efficiency — rather than as a sequence
of stage transitions.

`load_sleep_nights()` handles that shape and returns one row per night with
durations in hours. (`load_sleep_data()` is the counterpart for studies that
*do* carry stage episodes; it will raise on this data because the episode
columns are absent.)

In [ ]:
sleep_raw = read_sample("sleep_stage_summary")

nights = load_sleep_nights(sleep_raw)
nights.head()

## Using the CGM Sandbox

### A plain CGM tracing with overlays

Overlays each contribute one layer to the same time axis.

In [ ]:
viewer = CGMViewer(source="client", client_df=cgm_raw, gl_range=(0, 250))

viewer.add_overlay(TimeInRangeOverlay())
viewer.add_overlay(FoodEntryOverlay(source="client", client_df=food_raw))

viewer.show()

### Adding sleep: the multimodal view

This is where a second modality earns its place. `SleepWindowOverlay` shades
each night behind the glucose trace, and `SleepCompositionExtension` adds a
panel showing how each night was composed — deep, light, REM and awake.

Reading the two together answers a question glucose alone cannot: a rise at
02:00 and a rise at 14:00 look identical in a single trace.

In [ ]:
viewer = CGMViewer(source="client", client_df=cgm_raw, gl_range=(0, 250))

viewer.add_overlay(SleepWindowOverlay(nights))
viewer.add_overlay(TimeInRangeOverlay())
viewer.add_extensions(SleepCompositionExtension(nights))

viewer.show()

## What the pairing buys you

Splitting the glucose record by sleep state needs **both** datasets. Neither one
produces this on its own.

In [ ]:
nocturnal_vs_diurnal(cgm_raw, nights)

## A note on hypnograms

The sandbox also ships `HypnogramExtension`, which renders a stage trace across
the night. It needs per-episode timing (`sleep_stage_episodes_*`) and this study
does not carry it — hence the summary panel above rather than a hypnogram.

That is a property of the data, not a limitation of the viewer: point
`HypnogramExtension` at a study whose sleep records include episodes and it
draws the trace. It is worth checking the columns before reaching for it.